# Results, Discussion and Conclusion

## Research question and evaluation design

The primary analysis examines whether recent Tuen Mun Air Quality Health Index (AQHI) observations can identify whether AQHI will be **at least 4 six hours later**. The primary positive class is target AQHI >= 4. A secondary, preliminary analysis retains the stricter target AQHI >= 7 so that its limited performance can be compared with the main task.

Both targets use five classifiers—Decision Tree, Random Forest, Logistic Regression, K-Nearest Neighbours (KNN), and Support Vector Machine (SVM)—against a class-persistence baseline. The baseline carries the current AQHI classification forward as the six-hour prediction. Each target is evaluated with preceding AQHI history windows of 3, 6, or 12 hours. Model configurations and classification thresholds were selected separately for each target and history window.

For each target, all three history windows use the same matched complete-case cohort of 2,134 forecasts. The forecast input times run from 1 April 2026 00:00 to 30 June 2026 18:00 HKT; target times run from 1 April 2026 06:00 to 1 July 2026 00:00 HKT. The AQHI >= 4 cohort contains 581 positive targets (27.2%); the AQHI >= 7 cohort contains 34 (1.6%). The timestamp cohort is the same across both targets, while the positive-class definition differs.

The holdout period was inspected during model development. Accordingly, all test-set comparisons should be interpreted as exploratory evidence rather than as a fully independent final evaluation.

## Data and code workflow

The primary data are in `outputs/aqhi4-lookback-comparison/`; the secondary AQHI >= 7 comparison is in `outputs/model-evaluation/lookback_comparison/`. Its target definition is documented in `outputs/model-evaluation/model_settings.json`; the validation code independently checks every row against the observed AQHI threshold. The first code cell reads both sets of exported metrics and row-level predictions. It then:

1. Verifies each actual class against its target rule (AQHI >= 4 or AQHI >= 7), checks that all methods and history windows share the same target timestamps, and confirms the two targets use the same timestamps.
2. Recalculates TP, FN, FP, TN, precision, recall, F1, and accuracy from row-level predictions for both targets.
3. Compares all 36 method-window metric rows with the corresponding `test_metrics.csv` files.
4. Prints selected six-hour cases, including a target from Damon's first-event timeline, to illustrate the threshold comparison.

Run the first code cell before the case-extraction cell, or use Run All. In VS Code, open the notebook from the project root so the relative file paths resolve.


In [1]:
from pathlib import Path
import csv

TASKS = {
    "AQHI >= 4": {
        "threshold": 4,
        "folder": Path("outputs/aqhi4-lookback-comparison"),
    },
    "AQHI >= 7": {
        "threshold": 7,
        "folder": Path("outputs/model-evaluation/lookback_comparison"),
    },
}
METHODS = [
    "Decision Tree", "Random Forest", "Logistic Regression",
    "KNN", "SVM", "Persistence baseline"
]
LOOKBACKS = ["3h", "6h", "12h"]


def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


prediction_rows = {}
metrics_by_task = {}
recomputed = {}
cohort_by_task = {}

# Recompute every method/window row for both target definitions.
for task, info in TASKS.items():
    folder = info["folder"]
    required = ["test_metrics.csv"] + [f"test_predictions_{lb}.csv" for lb in LOOKBACKS]
    missing = [name for name in required if not (folder / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Missing files for {task}: {', '.join(missing)}")

    metrics_rows = read_csv(folder / "test_metrics.csv")
    metrics_by_task[task] = {(r["lookback"], r["method"]): r for r in metrics_rows}
    if len(metrics_by_task[task]) != len(LOOKBACKS) * len(METHODS):
        raise ValueError(f"Expected 18 metric rows for {task}; found {len(metrics_by_task[task])}.")

    prediction_rows[task] = {}
    cohort_by_method_window = {}
    for lookback in LOOKBACKS:
        records = read_csv(folder / f"test_predictions_{lookback}.csv")
        prediction_rows[task][lookback] = records
        for method in METHODS:
            selected = [r for r in records if r["method"] == method]
            if len(selected) != 2134:
                raise ValueError(f"Unexpected sample count for {task}, {lookback}, {method}: {len(selected)}")
            actual_by_time = {r["target_time"]: (float(r["actual_aqhi"]), r["actual_class"]) for r in selected}
            if len(actual_by_time) != len(selected):
                raise ValueError(f"Duplicate target timestamps for {task}, {lookback}, {method}.")
            cohort_by_method_window[(lookback, method)] = actual_by_time
            if not all(int(aqhi >= info["threshold"]) == int(actual_class) for aqhi, actual_class in actual_by_time.values()):
                raise ValueError(f"At least one actual_class label disagrees with {task}.")

            tp = sum(r["actual_class"] == "1" and r["predicted_class"] == "1" for r in selected)
            fn = sum(r["actual_class"] == "1" and r["predicted_class"] == "0" for r in selected)
            fp = sum(r["actual_class"] == "0" and r["predicted_class"] == "1" for r in selected)
            tn = sum(r["actual_class"] == "0" and r["predicted_class"] == "0" for r in selected)
            precision = tp / (tp + fp) if tp + fp else 0
            recall = tp / (tp + fn) if tp + fn else 0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
            accuracy = (tp + tn) / len(selected)
            recomputed[(task, lookback, method)] = {
                "samples": len(selected), "positive_samples": tp + fn,
                "tp": tp, "fn": fn, "fp": fp, "tn": tn,
                "precision": precision, "recall": recall,
                "f1": f1, "accuracy": accuracy,
            }

    reference = cohort_by_method_window[(LOOKBACKS[0], METHODS[0])]
    if any(values != reference for values in cohort_by_method_window.values()):
        raise ValueError(f"Methods or lookback windows do not share the same cases for {task}.")
    cohort_by_task[task] = reference

# The two thresholds must use the same observed AQHI values and target times.
aqhi_values_4 = {time: values[0] for time, values in cohort_by_task["AQHI >= 4"].items()}
aqhi_values_7 = {time: values[0] for time, values in cohort_by_task["AQHI >= 7"].items()}
if aqhi_values_4 != aqhi_values_7:
    raise ValueError("The two targets do not share the same test timestamps and observed AQHI values.")

count_fields = {
    "samples": "samples", "positive_samples": "positive_samples",
    "true_positive": "tp", "false_negative": "fn",
    "false_positive": "fp", "true_negative": "tn",
}
rate_fields = ("precision", "recall", "f1", "accuracy")
checks = []
for task, metrics in metrics_by_task.items():
    for (lookback, method), row in metrics.items():
        calc = recomputed[(task, lookback, method)]
        counts_match = all(int(row[field]) == calc[key] for field, key in count_fields.items())
        rates_match = all(abs(calc[field] - float(row[field])) < 1e-10 for field in rate_fields)
        checks.append(counts_match and rates_match)

if not all(checks):
    raise ValueError("Recalculated row-level predictions do not match the exported metric files.")

print(f"Metric rows checked: {len(checks)} / 36; all counts and rates match: {all(checks)}")
for task in TASKS:
    cohort = cohort_by_task[task]
    positive_count = sum(actual_class == "1" for _, actual_class in cohort.values())
    print(f"{task}: {len(cohort):,} shared forecast times; {positive_count} positive targets ({positive_count / len(cohort):.1%})")
print("AQHI >= 4 and AQHI >= 7 use the same timestamps and observed AQHI values: True")


Metric rows checked: 36 / 36; all counts and rates match: True
AQHI >= 4: 2,134 shared forecast times; 581 positive targets (27.2%)
AQHI >= 7: 2,134 shared forecast times; 34 positive targets (1.6%)
AQHI >= 4 and AQHI >= 7 use the same timestamps and observed AQHI values: True


## Metric definitions

Precision is TP / (TP + FP), the proportion of positive predictions that are correct. Recall is TP / (TP + FN), the proportion of positive targets detected under the threshold being evaluated. F1 is the harmonic mean of precision and recall. TP (true positive) is a correctly detected positive target; FN (false negative) is a missed positive target; FP (false positive) is an incorrect positive alert; and TN (true negative) is a correctly identified negative target.

## Preliminary results: target AQHI >= 7

The AQHI >= 7 experiment is retained as a preliminary, secondary analysis. On the shared 2,134 forecast times, only 34 targets (1.6%) meet AQHI >= 7. This sparse positive class means each missed target changes recall by about 2.9 percentage points; the results should therefore be interpreted cautiously.

The table reports the six-hour-history test results. Values are rounded to three decimal places.

| Method | Precision | Recall | F1 | TP | FN | FP |
|---|---:|---:|---:|---:|---:|---:|
| Decision Tree | 0.149 | 0.294 | 0.198 | 10 | 24 | 57 |
| Random Forest | 0.160 | 0.441 | 0.234 | 15 | 19 | 79 |
| Logistic Regression | 0.244 | 0.294 | **0.267** | 10 | 24 | 31 |
| KNN | 0.119 | 0.353 | 0.178 | 12 | 22 | 89 |
| SVM | 0.170 | **0.471** | 0.250 | 16 | 18 | 78 |
| Persistence baseline | 0.176 | 0.176 | 0.176 | 6 | 28 | 28 |

Among the selected six-hour configurations, Logistic Regression has the highest observed test F1 (0.267), while SVM detects the largest share of positive targets (recall 0.471; 16 of 34). Even SVM misses 18 positive targets and produces 78 false alerts. The persistence baseline misses 28 of 34. The maximum F1 by history window is 0.260 for Logistic Regression at 3h, 0.267 for Logistic Regression at 6h, and 0.250 for Random Forest at 12h; history length does not produce a consistent improvement.

![AQHI >= 7 test F1 across history windows](outputs/results-discussion/aqhi7_history_f1.png)

## Main results: target AQHI >= 4

For this main-results table, the positive class is specifically target AQHI >= 4.

The comparison evaluates six methods—five classifiers and the persistence baseline—across all three history windows. Values are rounded to three decimal places.


The table reports test-set metrics for all methods and lookback windows. Values are rounded to three decimal places.

| History | Method | Precision | Recall | F1 | TP | FN | FP |
|---|---|---:|---:|---:|---:|---:|---:|
| 3h | Decision Tree | 0.711 | 0.702 | 0.706 | 408 | 173 | 166 |
| 3h | Random Forest | 0.666 | 0.793 | 0.724 | 461 | 120 | 231 |
| 3h | Logistic Regression | 0.703 | 0.737 | 0.719 | 428 | 153 | 181 |
| 3h | KNN | 0.649 | 0.849 | 0.735 | 493 | 88 | 267 |
| 3h | SVM | 0.682 | 0.754 | 0.716 | 438 | 143 | 204 |
| 3h | Persistence baseline | 0.692 | 0.692 | 0.692 | 402 | 179 | 179 |
| 6h | Decision Tree | 0.578 | 0.921 | 0.710 | 535 | 46 | 390 |
| 6h | Random Forest | 0.681 | 0.812 | **0.741** | 472 | 109 | 221 |
| 6h | Logistic Regression | **0.711** | 0.728 | 0.719 | 423 | 158 | 172 |
| 6h | KNN | 0.642 | **0.840** | 0.728 | 488 | 93 | 272 |
| 6h | SVM | 0.680 | 0.771 | 0.723 | 448 | 133 | 211 |
| 6h | Persistence baseline | 0.692 | 0.692 | 0.692 | 402 | 179 | 179 |
| 12h | Decision Tree | **0.752** | 0.642 | 0.693 | 373 | 208 | 123 |
| 12h | Random Forest | 0.746 | 0.678 | 0.711 | 394 | 187 | 134 |
| 12h | Logistic Regression | 0.715 | 0.695 | 0.705 | 404 | 177 | 161 |
| 12h | KNN | 0.625 | 0.836 | 0.716 | 486 | 95 | 291 |
| 12h | SVM | 0.681 | 0.771 | 0.723 | 448 | 133 | 210 |
| 12h | Persistence baseline | 0.692 | 0.692 | 0.692 | 402 | 179 | 179 |

![Test F1 by history window](outputs/aqhi4-lookback-comparison/lookback_f1_comparison.png)

### Precision-recall trade-off at six hours

![Precision-recall plot for six-hour forecasts](outputs/results-discussion/precision_recall_6h.png)

Each point represents one method; the axes show precision and recall, and the label gives its F1. Random Forest has the strongest F1 balance. Decision Tree reaches higher recall but substantially lower precision.

### False positives and false negatives

![Six-hour false-positive and false-negative counts](outputs/results-discussion/error_counts_6h.png)

This plot makes the error trade-off explicit: Decision Tree misses fewer high-AQHI hours, but produces more false alerts.

### Six-hour results

Among the six-hour models, Random Forest achieved the highest F1 (0.741), an absolute increase of 0.049 over the persistence baseline (0.692). Its precision was 0.681 and recall was 0.812, with 472 true positives, 109 false negatives, and 221 false positives.

Decision Tree achieved the highest recall (0.921), detecting 535 of 581 positive target hours. However, it also produced 390 false positives and had the lowest six-hour precision (0.578). Logistic Regression had the highest six-hour precision (0.711), while KNN had the second-highest recall (0.840). Thus, model ranking depends on the relative cost assigned to missed high-AQHI hours and false alerts.

### Effect of lookback window

The effect of history length varies by method. Random Forest F1 increased from 0.724 with 3h history to 0.741 with 6h history. Logistic Regression was nearly unchanged between 3h and 6h (0.719 for both), while KNN performed best with 3h history (0.735) and declined to 0.728 at 6h and 0.716 at 12h. Decision Tree performed best at 6h (0.710). SVM performed similarly across windows, with F1 ranging from 0.716 to 0.723; its 12h result was only marginally above its six-hour result.

Overall, no single history window improved every classifier. The persistence baseline was constant across windows, as expected.


### Same six-hour cohort, two target definitions

The two thresholds use the same 2,134 target times, but define different positive classes: AQHI >= 4 has 581 positives (27.2%), while AQHI >= 7 has 34 (1.6%). The highest observed six-hour test F1 is from Random Forest for AQHI >= 4 and Logistic Regression for AQHI >= 7. Decision Tree has the highest recall for AQHI >= 4; SVM has the highest recall for AQHI >= 7.

| Target definition | Positive targets | Highest observed six-hour test F1 | Highest six-hour recall |
|---|---:|---|---|
| AQHI >= 4 | 581 / 2,134 (27.2%) | Random Forest: 0.741 (precision 0.681; recall 0.812) | Decision Tree: 0.921 |
| AQHI >= 7 | 34 / 2,134 (1.6%) | Logistic Regression: 0.267 (precision 0.244; recall 0.294) | SVM: 0.471 |

![Six-hour F1 for the two AQHI target definitions](outputs/results-discussion/threshold_f1_6h.png)

These F1 values describe separate classification tasks, not a like-for-like comparison of model quality: the positive class and its prevalence change with the target threshold. The AQHI >= 7 result is especially uncertain because it is based on only 34 positive hourly targets.


## Error analysis: selected six-hour forecasts

The examples below show selected six-hour predictions. Forecast issued is the input time; target time is six hours later. A prediction of 1 indicates that the method forecasts target AQHI >= 4.

| Forecast issued (HKT) | Target time (HKT) | Actual AQHI | Decision Tree | Random Forest | Logistic Regression | KNN | SVM | Persistence |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| 25 Apr 09:00 | 25 Apr 15:00 | 7 | 1 | 0 | 0 | 0 | 0 | 0 |
| 25 Apr 10:00 | 25 Apr 16:00 | 8 | 1 | 1 | 1 | 1 | 0 | 0 |
| 25 Apr 11:00 | 25 Apr 17:00 | 8 | 1 | 1 | 0 | 1 | 0 | 0 |
| 29 May 09:00 | 29 May 15:00 | 7 | 0 | 0 | 0 | 0 | 0 | 0 |
| 19 May 08:00 | 19 May 14:00 | 2 | 1 | 1 | 1 | 1 | 1 | 0 |

![Classification outcomes for selected six-hour forecast cases](outputs/results-discussion/selected_cases_6h.png)

In the heatmap, TP (true positive) marks a correctly detected AQHI >= 4 target; FN (false negative) marks a missed AQHI >= 4 target; FP (false positive) marks a false alert for a target below 4; and TN (true negative) marks a correctly identified target below 4.

On 25 April, the target AQHI was 8 at 16:00; Decision Tree, Random Forest, Logistic Regression, and KNN issued positive predictions, whereas SVM and the persistence baseline did not. At 15:00, when the target AQHI was 7, only Decision Tree issued a positive prediction. On 29 May, all six methods failed to flag the target AQHI of 7 at 15:00. On 19 May, all five classifiers predicted AQHI >= 4 for a target AQHI of 2, while the persistence baseline did not. These examples illustrate that aggregate metrics can conceal important differences in performance on individual hours.

### Same event under both target thresholds

Damon's first-event timeline shows a target AQHI of 8 at 17:00 HKT on 1 April 2026. The six-hour forecast was issued at 11:00 HKT. Because AQHI 8 is positive under both definitions, the outcomes show how predictions differ when the models are trained and tuned for each target: all five classifiers predicted positive for AQHI >= 4, while all five predicted negative for AQHI >= 7. The persistence baseline predicted negative for both.

| Target definition | Actual class at AQHI 8 | Five classifiers | Persistence baseline |
|---|---|---|---|
| AQHI >= 4 | Positive | All five predicted positive (TP) | Negative (FN) |
| AQHI >= 7 | Positive | All five predicted negative (FN) | Negative (FN) |

![Same six-hour event classified under both target thresholds](outputs/results-discussion/threshold_case_apr01_17.png)

This case is illustrative, not representative of all test hours. Adjacent hourly targets may belong to the same pollution episode, so timestamp-level rows should not be interpreted as independent events.


In [2]:
# Rebuild selected six-hour AQHI >= 4 examples and the paired threshold case.
case_targets = {
    "2026-04-25 15:00:00+08:00",
    "2026-04-25 16:00:00+08:00",
    "2026-04-25 17:00:00+08:00",
    "2026-05-29 15:00:00+08:00",
    "2026-05-19 14:00:00+08:00",
}
cases = {}
for r in prediction_rows["AQHI >= 4"]["6h"]:
    if r["target_time"] in case_targets:
        item = cases.setdefault(r["target_time"], {
            "forecast_issued": r["timestamp"],
            "actual_aqhi": r["actual_aqhi"],
            "predictions": {}
        })
        item["predictions"][r["method"]] = r["predicted_class"]

if set(cases) != case_targets or any(
    set(case["predictions"]) != set(METHODS) for case in cases.values()
):
    raise ValueError("Selected AQHI >= 4 case timestamps or predictions are incomplete.")

for target in sorted(cases):
    item = cases[target]
    print(target, "AQHI", item["actual_aqhi"],
          " ".join(f"{m}={item['predictions'][m]}" for m in METHODS))

# One timestamp from Damon's first-event timeline, evaluated under both targets.
paired_target = "2026-04-01 17:00:00+08:00"
paired_case = {}
for task in TASKS:
    rows = [r for r in prediction_rows[task]["6h"] if r["target_time"] == paired_target]
    if len(rows) != len(METHODS) or {float(r["actual_aqhi"]) for r in rows} != {8.0}:
        raise ValueError(f"Paired threshold example is incomplete for {task}.")
    paired_case[task] = {r["method"]: r["predicted_class"] for r in rows}
print("Paired threshold case:", paired_target, "actual AQHI=8; issued six hours earlier")
for task, predictions in paired_case.items():
    print(task, " ".join(f"{m}={predictions[m]}" for m in METHODS))


2026-04-25 15:00:00+08:00 AQHI 7.0 Decision Tree=1 Random Forest=0 Logistic Regression=0 KNN=0 SVM=0 Persistence baseline=0
2026-04-25 16:00:00+08:00 AQHI 8.0 Decision Tree=1 Random Forest=1 Logistic Regression=1 KNN=1 SVM=0 Persistence baseline=0
2026-04-25 17:00:00+08:00 AQHI 8.0 Decision Tree=1 Random Forest=1 Logistic Regression=0 KNN=1 SVM=0 Persistence baseline=0
2026-05-19 14:00:00+08:00 AQHI 2.0 Decision Tree=1 Random Forest=1 Logistic Regression=1 KNN=1 SVM=1 Persistence baseline=0
2026-05-29 15:00:00+08:00 AQHI 7.0 Decision Tree=0 Random Forest=0 Logistic Regression=0 KNN=0 SVM=0 Persistence baseline=0
Paired threshold case: 2026-04-01 17:00:00+08:00 actual AQHI=8; issued six hours earlier
AQHI >= 4 Decision Tree=1 Random Forest=1 Logistic Regression=1 KNN=1 SVM=1 Persistence baseline=0
AQHI >= 7 Decision Tree=0 Random Forest=0 Logistic Regression=0 KNN=0 SVM=0 Persistence baseline=0


## Discussion

The six-hour Random Forest provides the strongest balance of precision and recall among the evaluated six-hour methods, as measured by F1. It detects 472 of 581 positive target hours and produces 221 false alerts. The six-hour Decision Tree may be preferable when reducing missed positive hours is the primary objective: it misses 46 positive hours, compared with 109 for Random Forest, but produces 390 false alerts. The appropriate choice therefore depends on the intended use and the relative consequences of missed alerts and false alarms.

The AQHI >= 4 lookback comparison does not support a general claim that more historical data improves performance. Random Forest benefits from extending history from 3h to 6h, KNN performs best with 3h, and SVM varies little across the three windows. The AQHI >= 7 results also show no consistent history-window winner. Parameters and classification thresholds were selected separately for each target, method, and history window, so these are comparisons of tuned configurations rather than an isolated test of adding lag features.

The threshold comparison must be interpreted as a comparison of two different tasks. AQHI >= 7 has only 34 positives in the shared test cohort, compared with 581 for AQHI >= 4. Its lower recall and F1 are consistent with a much rarer positive class, but the small number of positives also makes these estimates less stable. The paired 1 April case illustrates this difference without establishing that one target definition is universally preferable.

## Limitations

- **Geographic and temporal coverage:** The evaluation uses one monitoring station and a three-month test period (April–June 2026). Performance may differ at other locations or seasons; later periods and other stations should be evaluated.
- **Hourly dependence:** Consecutive target hours may belong to the same pollution episode and are autocorrelated. The reported hourly counts should not be interpreted as counts of independent pollution events. Future work should add episode-level detection and lead-time measures.
- **Rare AQHI >= 7 targets:** Only 34 of 2,134 test targets meet AQHI >= 7. One additional missed positive changes recall by about 2.9 percentage points, so its metrics are sensitive to a small number of cases. More years of observations and a later untouched test period are needed for a more stable evaluation.
- **Input coverage:** Models use AQHI history and time-of-day features; meteorological, emissions, and other potentially informative variables are not included. Future models could add meteorological and emissions data.
- **Missing and substituted readings:** The comparison follows the repository's complete-case cohort and flagged-reading policy. Missing hours were not interpolated. A sensitivity analysis under alternative flagged-reading policies would help test the effect of this choice.
- **Exploratory holdout:** The test period had been inspected during model development. These results may be optimistic and should be confirmed on a later, untouched period.
- **Operational evaluation:** The analysis reports hour-level classification metrics. A future evaluation should also consider episode-level detection, warning lead time, and the burden of repeated false alerts.

## Conclusion

1. **Primary AQHI >= 4 result:** Random Forest with six-hour history achieved the highest observed six-hour test F1 (0.741; precision 0.681; recall 0.812), exceeding the persistence baseline F1 (0.692) by 0.049.
2. **Detection versus false alerts:** Decision Tree had the highest AQHI >= 4 recall (0.921), but also produced 390 false positives. Model choice depends on the cost of missed targets versus false alerts.
3. **Secondary AQHI >= 7 result:** Only 34 targets were positive. Logistic Regression had the highest observed six-hour test F1 (0.267), while SVM had the highest recall (0.471); these preliminary estimates are limited by the small positive count.
4. **History-window effect:** No history window consistently improved every method.
5. **Scope of inference:** Findings apply to the matched Tuen Mun test cohort and remain exploratory because the holdout period had already been inspected.

On this test cohort, six-hour Random Forest is the most balanced candidate among the evaluated configurations for identifying AQHI >= 4 six hours ahead. The AQHI >= 7 analysis is a secondary result with limited positive cases, not a replacement for the primary finding. Both conclusions are provisional; evaluation on a later untouched period, with event-level measures and broader predictive inputs, is required before operational conclusions can be drawn.
